Merge all the subreddit datasets & aggregate based on location and day

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
BASE_DIR = Path.cwd()

file_path1 = BASE_DIR / "Europetravel_processed_with_locations2.xlsx"
file_path2 = BASE_DIR / "UKtravel_processed_with_locations.xlsx"
file_path3 = BASE_DIR / "travel_processed_with_locations.xlsx"
file_path4 = BASE_DIR / "backpacking_processed_with_locations.xlsx"
file_path5 = BASE_DIR / "solotravel_processed_with_locations.xlsx"

# print("File location:", file_path)
# print("File exists:", file_path.exists())

df1 = pd.read_excel(file_path1)
df2 = pd.read_excel(file_path2)
df3 = pd.read_excel(file_path3)
df4 = pd.read_excel(file_path4)
df5 = pd.read_excel(file_path5)
# df1.head()

In [3]:
reddit = pd.concat([df1, df2, df3, df4, df5], ignore_index=True)
reddit.shape
# reddit.head()

(407950, 20)

In [4]:
import numpy as np

# Use country as destination
reddit["destination"] = reddit["country"]

# Log-weighted engagement
reddit["engagement_score"] = np.log1p(reddit["engagement"])

# Weighted sentiment (engagement * sentiment strength)
reddit["weighted_sentiment"] = (
    reddit["engagement_score"] * reddit["sentiment_compound"]
)

In [5]:
reddit_agg = (
    reddit
    .groupby(["day", "destination", "sentiment_label"])
    .agg({
        "engagement": "sum",               # raw engagement
        "engagement_score": "sum",         # weighted engagement
        "weighted_sentiment": "sum"        # weighted sentiment
    })
    .reset_index()
)



In [6]:
reddit_agg.shape
reddit_agg.to_csv('reddit.csv', index=False)

In [7]:
reddit_positive = reddit_agg[
    reddit_agg["sentiment_label"] == "positive"
].copy()

reddit_negative = reddit_agg[
    reddit_agg["sentiment_label"] == "negative"
].copy()

reddit_neutral = reddit_agg[
    reddit_agg["sentiment_label"] == "neutral"
].copy()

In [8]:
reddit_pivot = (
    reddit_agg
    .pivot_table(
        index=["day", "destination"],
        columns="sentiment_label",
        values=["engagement", "engagement_score", "weighted_sentiment"],
        fill_value=0
    )
)

reddit_pivot.columns = [
    f"{metric}_{sentiment}" 
    for metric, sentiment in reddit_pivot.columns
]

reddit_pivot = reddit_pivot.reset_index()